
# NOVA CORPORATION | BR001
## DATA QUALITY — STATISTICAL OUTLIER REVIEW

**Business Request:** BR001 — Executive Performance Review  
**Company:** Nova Corporation (Synthetic Business Simulation)  
**Reporting Period:** H1 2025 vs. H1 2026  
**DQ Control:** DQ013 — Statistical Outlier Assessment  
**Source:** BR001 Validated Analytical Extract  
**Technology:** Python / pandas  
**Master Register:** `Nova_BR001_Data_Quality_Workbook.xlsx`  
**Execution Policy:** READ ONLY

---

## 00 | Overview & Governance

### PURPOSE

Investigate statistical outliers in transactional and financial measures to determine whether unusual observations represent plausible business activity or potential data-quality issues.

### METHODOLOGY

01. Environment & Data Loading
02. Source Reconciliation
03. Exploratory Analysis
04. Statistical Outlier Detection
05. Anomaly Investigation
06. High-Cost Loss Exposure
07. Product-Level Loss Concentration
08. Investigation Conclusion

### GOVERNANCE

- Statistical outliers are not automatically data-quality defects.
- Source records remain unchanged.
- No observations are removed, capped, or corrected.
- Technical evidence supports the master Data Quality workbook.
- Business clarification is required before remediation decisions.

### TRACEABILITY

**Control:** DQ013  
**Master Documentation:** BR001 Data Quality Workbook  
**Supporting Evidence:** Python statistical investigation  
**Downstream Process:** Business Clarification

---



## 01 | Environment & Data Loading

**Objective:** Load the validated BR001 analytical extract and confirm that the expected project structure and source file are available.

### SOURCE INFORMATION

- Database: `nova_analytics_warehouse.db`
- Analytical Extract: `br001_raw_analytical_extract.csv`
- Expected Population: 23,530 sales lines
- Reporting Periods: H1 2025 and H1 2026

### METHODOLOGY

1. Locate the Nova Corporation project directory.
2. Verify the analytical extract exists.
3. Load the source file into pandas.
4. Preserve transaction identifiers as strings.
5. Inspect the dataset structure and sample records.

### GOVERNANCE

The analytical extract is loaded for investigation only.

No source records are modified.


In [15]:

# ============================================================
# NOVA CORPORATION | BR001
# 01 | ENVIRONMENT & DATA LOADING
# ============================================================

from pathlib import Path
import pandas as pd


# ------------------------------------------------------------
# 01.1 | PROJECT ROOT RESOLUTION
# ------------------------------------------------------------

def find_project_root(start_path=None):
    start = Path(start_path or Path.cwd()).resolve()

    for candidate in (start, *start.parents):
        required_paths = [
            candidate / "03_sql",
            candidate / "04_data",
            candidate / "05_analysis",
        ]

        if all(path.is_dir() for path in required_paths):
            return candidate

    raise FileNotFoundError(
        "Nova_Corporation project root was not found. "
        "Open VS Code from the project folder and retry."
    )


PROJECT_ROOT = find_project_root()


# ------------------------------------------------------------
# 01.2 | SOURCE FILE VALIDATION
# ------------------------------------------------------------

source_path = (
    PROJECT_ROOT
    / "04_data"
    / "raw"
    / "br001_raw_analytical_extract.csv"
)

if not source_path.is_file():
    raise FileNotFoundError(
        f"Source CSV not found: {source_path}"
    )

print("Project root:", PROJECT_ROOT)
print("Source CSV:", source_path)
print("Source path validation: PASS")


# ------------------------------------------------------------
# 01.3 | DATA LOADING
# ------------------------------------------------------------

df = pd.read_csv(
    source_path,
    dtype={
        "sales_line_id": "string",
        "order_id": "string"
    }
)


# ------------------------------------------------------------
# 01.4 | INITIAL INSPECTION
# ------------------------------------------------------------

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())


Project root: Nova_Corporation
Source CSV: Nova_Corporation\04_data\raw\br001_raw_analytical_extract.csv
Source path validation: PASS
Rows: 23530
Columns: 36


,sales_line_id,order_id,order_date,year,half_year,quarter,month_number,month_name,year_month,customer_id,...,unit_price,unit_cost,discount_pct,gross_revenue,discount_amount,net_revenue,total_cost,gross_profit,payment_method,order_status
0,SL0000226,O0000151,2025-01-01,2025,H1,1,1,January,2025-01,C01758,...,287.13,168.21,0.00,287.13,0.00,287.13,168.21,118.92,Debit Card,Completed
1,SL0000251,O0000169,2025-01-01,2025,H1,1,1,January,2025-01,C01129,...,189.69,105.42,0.05,189.69,9.48,180.21,105.42,74.79,Debit Card,Completed
2,SL0000530,O0000338,2025-01-01,2025,H1,1,1,January,2025-01,C00078,...,762.20,326.47,0.10,762.20,76.22,685.98,326.47,359.51,Cash,Completed
3,SL0000681,O0000442,2025-01-01,2025,H1,1,1,January,2025-01,C00533,...,395.45,224.86,0.30,1581.80,474.54,1107.26,899.44,207.82,Debit Card,Completed
4,SL0000842,O0000544,2025-01-01,2025,H1,1,1,January,2025-01,C02470,...,534.53,384.88,0.10,534.53,53.45,481.08,384.88,96.20,Debit Card,Completed



---

## 02 | Source Reconciliation

**Objective:** Verify that the Python dataset reconciles with the previously validated SQL extraction.

**Reference:** `04_extraction_validation.sql`  
**Source:** BR001 Analytical Extract  
**Master Documentation:** BR001 Data Quality Workbook

### VALIDATION CONTROLS

- Total sales-line population
- Gross Revenue
- Discount Amount
- Net Revenue
- Total Cost
- Gross Profit

### ACCEPTANCE CRITERIA

Each financial measure must reconcile with the SQL baseline within the established tolerance of 0.01.

The record count must match the expected population exactly.

### GOVERNANCE

A successful reconciliation confirms consistency with the extraction baseline, not the absence of underlying data-quality issues.

---


In [16]:

# ============================================================
# NOVA CORPORATION | BR001
# 02 | SOURCE RECONCILIATION
# ============================================================

# ------------------------------------------------------------
# 02.1 | VALIDATED SQL BASELINE
# ------------------------------------------------------------

expected = {
    "rows": 23530,
    "gross_revenue": 17683230.67,
    "discount_amount": 2745693.85,
    "net_revenue": 14937536.82,
    "total_cost": 10442161.65,
    "gross_profit": 4495375.17
}


# ------------------------------------------------------------
# 02.2 | PYTHON VS SQL RECONCILIATION
# ------------------------------------------------------------

results = []

for metric, expected_value in expected.items():

    actual_value = (
        len(df) if metric == "rows"
        else round(df[metric].sum(), 2)
    )

    difference = round(actual_value - expected_value, 2)

    results.append({
        "Metric": metric,
        "SQL Baseline": expected_value,
        "Python Result": actual_value,
        "Difference": difference,
        "Status": (
            "PASS"
            if abs(difference) < 0.01
            else "FAIL"
        )
    })


# ------------------------------------------------------------
# 02.3 | VALIDATION RESULTS
# ------------------------------------------------------------

validation = pd.DataFrame(results)

display(validation)


,Metric,SQL Baseline,Python Result,Difference,Status
0,rows,23530.00,23530.00,0.0,PASS
1,gross_revenue,17683230.67,17683230.67,0.0,PASS
2,discount_amount,2745693.85,2745693.85,0.0,PASS
3,net_revenue,14937536.82,14937536.82,0.0,PASS
4,total_cost,10442161.65,10442161.65,0.0,PASS
5,gross_profit,4495375.17,4495375.17,0.0,PASS



---

## 03 | Exploratory Analysis

**Objective:** Understand the distribution, central tendency, and variability of the transactional and financial measures before applying statistical outlier detection.

**DQ Reference:** DQ013  
**Method:** Descriptive Statistics

### VARIABLES

- Quantity
- Unit Price
- Unit Cost
- Discount Percentage
- Gross Revenue
- Discount Amount
- Net Revenue
- Total Cost
- Gross Profit

### METHODOLOGY

Calculate descriptive statistics, including:

1. Observation count
2. Mean and standard deviation
3. Minimum and maximum
4. First quartile (Q1)
5. Median (Q2)
6. Third quartile (Q3)

### INTERPRETATION

Descriptive statistics provide an initial understanding of the population.

Extreme values require further investigation before classification.

---


In [17]:

# ============================================================
# NOVA CORPORATION | BR001
# 03 | EXPLORATORY ANALYSIS
# ============================================================

# ------------------------------------------------------------
# 03.1 | NUMERIC VARIABLES
# ------------------------------------------------------------

numeric_columns = [
    "quantity",
    "unit_price",
    "unit_cost",
    "discount_pct",
    "gross_revenue",
    "discount_amount",
    "net_revenue",
    "total_cost",
    "gross_profit"
]


# ------------------------------------------------------------
# 03.2 | DESCRIPTIVE STATISTICS
# ------------------------------------------------------------

descriptive_stats = (
    df[numeric_columns]
    .describe()
    .T
)

display(descriptive_stats.round(2))


,count,mean,std,min,25%,50%,75%,max
quantity,23530.0,2.11,1.95,-2.00,1.00,1.00,3.00,21.00
unit_price,23530.0,358.04,212.44,8.08,183.24,326.26,516.79,916.94
unit_cost,23530.0,211.15,117.49,-4.75,110.12,198.14,317.10,452.91
discount_pct,23530.0,0.16,0.11,0.00,0.05,0.15,0.25,1.15
gross_revenue,23530.0,751.52,906.27,-1005.40,254.05,477.20,843.32,14688.36
discount_amount,23530.0,116.69,185.92,0.00,18.07,58.02,134.31,2680.10
net_revenue,23530.0,634.83,777.52,-1005.40,206.35,396.63,712.12,12485.11
total_cost,23530.0,443.78,523.74,-660.62,153.77,294.77,482.00,7934.43
gross_profit,23530.0,191.05,303.70,-494.87,32.07,99.51,224.00,5229.67



---

## 04 | Statistical Outlier Detection

**Objective:** Identify statistical outlier candidates using the Interquartile Range method.

**DQ Reference:** DQ013  
**Method:** IQR — 1.5 Multiplier  
**Source:** BR001 Analytical Extract

### METHODOLOGY

For each numeric variable:

1. Calculate the first quartile (Q1).
2. Calculate the third quartile (Q3).
3. Calculate IQR = Q3 − Q1.
4. Determine the lower boundary.
5. Determine the upper boundary.
6. Flag observations outside the boundaries.
7. Calculate the number and percentage of outliers.

### STATISTICAL RULE

**Lower Bound:** Q1 − (1.5 × IQR)

**Upper Bound:** Q3 + (1.5 × IQR)

An observation is flagged when its value falls below the lower boundary or exceeds the upper boundary.

### GOVERNANCE

- Outliers are statistical investigation candidates.
- The IQR method does not establish business validity.
- No records are removed or modified.
- Results must be reconciled with the master DQ workbook.

---


In [18]:

# ============================================================
# NOVA CORPORATION | BR001
# 04 | STATISTICAL OUTLIER DETECTION
# ============================================================

# ------------------------------------------------------------
# 04.1 | IQR CALCULATION
# ------------------------------------------------------------

iqr_results = []

for column in numeric_columns:

    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_mask = (
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    )

    iqr_results.append({
        "Variable": column,
        "Lower Bound": round(lower_bound, 2),
        "Upper Bound": round(upper_bound, 2),
        "Outlier Count": int(outlier_mask.sum()),
        "Outlier %": round(outlier_mask.mean() * 100, 2)
    })


# ------------------------------------------------------------
# 04.2 | OUTLIER DETECTION SUMMARY
# ------------------------------------------------------------

iqr_summary = pd.DataFrame(iqr_results)

display(iqr_summary)


,Variable,Lower Bound,Upper Bound,Outlier Count,Outlier %
0,quantity,-2.00,6.00,997,4.24
1,unit_price,-317.07,1017.10,0,0.00
2,unit_cost,-200.36,627.58,0,0.00
3,discount_pct,-0.25,0.55,1,0.00
4,gross_revenue,-629.85,1727.22,2373,10.08
5,discount_amount,-156.29,308.68,2059,8.75
6,net_revenue,-552.29,1470.76,2328,9.89
7,total_cost,-338.57,974.34,2611,11.10
8,gross_profit,-255.82,511.88,2054,8.73


In [29]:
# DQ013 — Retrieve original Total Cost IQR thresholds

q1 = df["total_cost"].quantile(0.25)
q3 = df["total_cost"].quantile(0.75)

iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print("Population:", len(df))
print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

print(
    "High Total Cost Outliers:",
    (df["total_cost"] > upper_bound).sum()
)

Population: 23530
Q1: 153.7725
Q3: 482.0
IQR: 328.22749999999996
Lower Bound: -338.5687499999999
Upper Bound: 974.34125
High Total Cost Outliers: 2610



---

## 05 | Anomaly Investigation

**Objective:** Investigate statistical outlier populations to understand their distribution, business characteristics, and potential financial implications.

**DQ Control:** DQ013 — Statistical Outlier Assessment  
**Source:** BR001 Analytical Extract  
**Methodology:** IQR-Based Detection & Business Segmentation  
**Master Register:** `Nova_BR001_Data_Quality_Workbook.xlsx`

### INVESTIGATION SCOPE

This section examines statistical anomalies through the following analytical components:

1. **Quantity Outlier Distribution**
   - Examine unusually low and high transaction quantities.
   - Identify the frequency of flagged values.

2. **Commercial Segmentation**
   - Analyze outlier concentration by sales channel, product category, and transaction status.
   - Compare outlier rates across business segments.

3. **Automated Outlier Profiling**
   - Apply consistent IQR criteria across numeric variables.
   - Generate reusable statistical profiles.
   - Identify segments with elevated outlier concentrations.

4. **High-Cost Transaction Investigation**
   - Isolate transactions exceeding the total-cost IQR upper boundary.
   - Compare their financial characteristics with the overall population.

5. **Financial Consistency Assessment**
   - Examine gross profit behavior within high-cost transactions.
   - Identify positive, zero, and negative gross profit cases.

6. **Negative Gross Profit Investigation**
   - Isolate high-cost transactions reporting negative gross profit.
   - Examine quantities, discounts, and recorded financial measures.
   - Prepare the affected population for detailed financial exposure analysis.

### METHODOLOGICAL CONSIDERATIONS

- IQR boundaries are calculated from the complete analytical population.
- Quantity-specific investigations may distinguish upper-tail observations from the complete two-sided outlier population.
- Outlier rates are calculated relative to their corresponding business segments.
- Statistical results are descriptive and do not independently establish business-rule violations.

### GOVERNANCE

- Source records remain unchanged.
- No outliers are automatically removed, capped, or corrected.
- Negative gross profit does not automatically indicate a data-quality defect.
- Statistical findings must be supported by reproducible evidence.
- Business interpretation and remediation decisions require authorized clarification.
- The master Data Quality workbook remains the official record of findings, statuses, and decisions.

### EXPECTED OUTPUTS

- Quantity outlier frequency distribution.
- Outlier rates by business dimension.
- Consolidated statistical anomaly profiles.
- High-cost transaction population.
- Financial consistency assessment.
- High-cost negative-profit population for subsequent investigation.

---


In [19]:

# ============================================================
# NOVA CORPORATION | BR001
# 05 | ANOMALY INVESTIGATION
# 05.1 | QUANTITY OUTLIER DISTRIBUTION
# ============================================================

# Identify quantity outliers using the established thresholds.
quantity_outliers = df[
    (df["quantity"] < -2) |
    (df["quantity"] > 6)
].copy()

# Frequency distribution of flagged quantities.
quantity_frequency = (
    quantity_outliers["quantity"]
    .value_counts()
    .sort_index()
    .rename_axis("Quantity")
    .reset_index(name="Records")
)

display(quantity_frequency)

print("Total quantity outliers:", len(quantity_outliers))


,Quantity,Records
0,7,366
1,8,236
2,9,139
3,10,92
4,11,54
5,12,36
6,13,25
7,14,15
8,15,10
9,16,9


Total quantity outliers: 997


In [20]:

# ============================================================
# 05.2 | QUANTITY OUTLIERS BY BUSINESS DIMENSION
# ============================================================

dimensions = [
    "channel_name",
    "category",
    "order_status"
]

for dimension in dimensions:
    summary = (
        df.assign(
            Quantity_Outlier=df["quantity"] > 6
        )
        .groupby(dimension, dropna=False)
        .agg(
            Total_Records=("quantity", "size"),
            Outlier_Records=("Quantity_Outlier", "sum")
        )
    )

    summary["Outlier_Rate_%"] = (
        summary["Outlier_Records"]
        / summary["Total_Records"] * 100
    ).round(2)

    print(f"\nQuantity outliers by {dimension}")

    display(
        summary.sort_values(
            "Outlier_Rate_%",
            ascending=False
        )
    )



Quantity outliers by channel_name


,Total_Records,Outlier_Records,Outlier_Rate_%
channel_name,,,
Telephone Sales,1652,74,4.48
Marketplace,3917,168,4.29
E-commerce,7435,317,4.26
Retail Store,7241,304,4.20
Corporate Sales,3284,134,4.08
NaN,1,0,0.00



Quantity outliers by category


,Total_Records,Outlier_Records,Outlier_Rate_%
category,,,
Home,4794,229,4.78
Technology,4710,202,4.29
Accessories,4666,196,4.20
Office,4700,196,4.17
Wellness,4660,174,3.73



Quantity outliers by order_status


,Total_Records,Outlier_Records,Outlier_Rate_%
order_status,,,
Pending,343,21,6.12
Partially Returned,1075,46,4.28
Completed,21345,901,4.22
Cancelled,767,29,3.78


In [21]:

# ============================================================
# 05.3 | REUSABLE OUTLIER PROFILING FUNCTION
# ============================================================

def analyze_outliers_by_segment(data, variable, dimension):
    """Profile IQR outliers across a business dimension."""

    q1 = data[variable].quantile(0.25)
    q3 = data[variable].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    flagged = (
        (data[variable] < lower_bound) |
        (data[variable] > upper_bound)
    )

    summary = (
        data.assign(Is_Outlier=flagged)
        .groupby(dimension, dropna=False)
        .agg(
            Total_Records=("Is_Outlier", "size"),
            Outlier_Records=("Is_Outlier", "sum")
        )
    )

    summary["Outlier_Rate_%"] = (
        summary["Outlier_Records"]
        / summary["Total_Records"] * 100
    ).round(2)

    return summary.sort_values(
        "Outlier_Rate_%",
        ascending=False
    )


# Reproduce the previously validated channel analysis.
display(
    analyze_outliers_by_segment(
        df,
        variable="quantity",
        dimension="channel_name"
    )
)


,Total_Records,Outlier_Records,Outlier_Rate_%
channel_name,,,
Telephone Sales,1652,74,4.48
Marketplace,3917,168,4.29
E-commerce,7435,317,4.26
Retail Store,7241,304,4.20
Corporate Sales,3284,134,4.08
NaN,1,0,0.00


In [22]:

# ============================================================
# 05.4 | AUTOMATED OUTLIER PROFILING
# ============================================================

business_dimensions = [
    "channel_name",
    "category",
    "order_status"
]

outlier_profiles = {}

for variable in numeric_columns:
    for dimension in business_dimensions:
        outlier_profiles[(variable, dimension)] = (
            analyze_outliers_by_segment(
                df, variable, dimension
            )
        )

print("Profiles generated:", len(outlier_profiles))

# Example: gross profit outliers by product category.
display(
    outlier_profiles[("gross_profit", "category")]
)


Profiles generated: 27


,Total_Records,Outlier_Records,Outlier_Rate_%
category,,,
Office,4700,448,9.53
Home,4794,456,9.51
Technology,4710,402,8.54
Accessories,4666,376,8.06
Wellness,4660,372,7.98


In [23]:

# ============================================================
# 05.5 | CONSOLIDATED OUTLIER PRIORITIZATION
# ============================================================

profile_tables = []

for (variable, dimension), profile in outlier_profiles.items():
    result = profile.reset_index().rename(
        columns={dimension: "Segment"}
    )

    result.insert(0, "Variable", variable)
    result.insert(1, "Dimension", dimension)

    profile_tables.append(result)

outlier_overview = pd.concat(
    profile_tables,
    ignore_index=True
)

# Exclude small segments from the initial ranking.
priority_review = (
    outlier_overview[
        outlier_overview["Total_Records"] >= 100
    ]
    .sort_values(
        "Outlier_Rate_%",
        ascending=False
    )
    .head(15)
)

display(priority_review)


,Variable,Dimension,Segment,Total_Records,Outlier_Records,Outlier_Rate_%
111,total_cost,category,Technology,4710,605,12.85
112,total_cost,category,Home,4794,589,12.29
66,gross_revenue,category,Home,4794,566,11.81
105,total_cost,channel_name,Marketplace,3917,455,11.62
106,total_cost,channel_name,Corporate Sales,3284,376,11.45
67,gross_revenue,category,Technology,4710,536,11.38
96,net_revenue,category,Technology,4710,529,11.23
97,net_revenue,category,Home,4794,536,11.18
113,total_cost,category,Wellness,4660,520,11.16
116,total_cost,order_status,Completed,21345,2378,11.14


In [24]:

# ============================================================
# 05.6 | HIGH-COST TRANSACTION INVESTIGATION
# ============================================================

cost_q1 = df["total_cost"].quantile(0.25)
cost_q3 = df["total_cost"].quantile(0.75)

cost_upper_bound = (
    cost_q3 + 1.5 * (cost_q3 - cost_q1)
)

high_cost = df[
    df["total_cost"] > cost_upper_bound
]

comparison = pd.DataFrame({
    "All Records": df[
        [
            "quantity",
            "gross_revenue",
            "net_revenue",
            "total_cost",
            "gross_profit"
        ]
    ].median(),

    "High-Cost Records": high_cost[
        [
            "quantity",
            "gross_revenue",
            "net_revenue",
            "total_cost",
            "gross_profit"
        ]
    ].median()
})

comparison["Ratio"] = (
    comparison["High-Cost Records"]
    / comparison["All Records"]
)

display(comparison.round(2))

print("High-cost records:", len(high_cost))


,All Records,High-Cost Records,Ratio
quantity,1.00,5.00,5.00
gross_revenue,477.20,2432.94,5.10
net_revenue,396.63,2039.36,5.14
total_cost,294.77,1423.95,4.83
gross_profit,99.51,588.72,5.92


High-cost records: 2610


In [25]:

# ============================================================
# 05.7 | HIGH-COST FINANCIAL CONSISTENCY
# ============================================================

high_cost_review = pd.DataFrame({
    "Metric": [
        "High-cost records",
        "Negative gross profit",
        "Zero gross profit",
        "Positive gross profit",
        "Negative gross profit rate (%)"
    ],
    "Value": [
        len(high_cost),
        int((high_cost["gross_profit"] < 0).sum()),
        int((high_cost["gross_profit"] == 0).sum()),
        int((high_cost["gross_profit"] > 0).sum()),
        round(
            (high_cost["gross_profit"] < 0).mean() * 100,
            2
        )
    ]
})

display(high_cost_review)


,Metric,Value
0,High-cost records,2610.00
1,Negative gross profit,190.00
2,Zero gross profit,0.00
3,Positive gross profit,2420.00
4,Negative gross profit rate (%),7.28


In [26]:

# ============================================================
# 05.8 | NEGATIVE GROSS PROFIT INVESTIGATION
# ============================================================

loss_cases = high_cost.loc[
    high_cost["gross_profit"] < 0
].copy()

loss_review = pd.DataFrame({
    "Metric": [
        "High-cost loss records",
        "Minimum gross profit",
        "Median gross profit",
        "Minimum quantity",
        "Maximum quantity",
        "Minimum discount percentage",
        "Maximum discount percentage"
    ],
    "Value": [
        len(loss_cases),
        loss_cases["gross_profit"].min(),
        loss_cases["gross_profit"].median(),
        loss_cases["quantity"].min(),
        loss_cases["quantity"].max(),
        loss_cases["discount_pct"].min(),
        loss_cases["discount_pct"].max()
    ]
})

display(loss_review)


,Metric,Value
0,High-cost loss records,190.00
1,Minimum gross profit,-479.34
2,Median gross profit,-95.22
3,Minimum quantity,3.00
4,Maximum quantity,13.00
5,Minimum discount percentage,0.15
6,Maximum discount percentage,0.40



---

## 06 | High-Cost Loss Exposure & Commercial Concentration

**Objective:** Quantify the financial exposure associated with high-cost sales lines reporting negative gross profit.

**DQ Control:** DQ013  
**Source:** BR001 Analytical Extract  
**Master Register:** `Nova_BR001_Data_Quality_Workbook.xlsx`

### METHODOLOGY

1. Validate the previously investigated high-cost population.
2. Calculate revenue, cost, gross profit, and margin.
3. Quantify the share of affected records and revenue.
4. Evaluate concentration by product category, sales channel, and transaction status.
5. Reconcile results against the established investigation baseline.

### VALIDATION BASELINE

- High-cost sales lines: 2,610
- High-cost negative-profit lines: 190

### GOVERNANCE

- Negative gross profit indicates potential commercial risk.
- Financial exposure does not automatically represent a data-quality defect.
- No source records are modified.
- Business validation remains necessary before remediation.


In [27]:

# ============================================================
# NOVA CORPORATION | BR001
# 06 | HIGH-COST LOSS EXPOSURE & CONCENTRATION
# ============================================================

# Reuses: df, high_cost, loss_cases
# Source data remains unchanged.

# ------------------------------------------------------------
# 06.1 | REQUIRED COLUMN VALIDATION
# ------------------------------------------------------------

required_columns = [
    "category",
    "channel_name",
    "order_status",
    "net_revenue",
    "total_cost",
    "gross_profit"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


# ------------------------------------------------------------
# 06.2 | POPULATION RECONCILIATION
# ------------------------------------------------------------

assert len(high_cost) == 2610, (
    f"Unexpected high-cost population: {len(high_cost)}"
)

assert len(loss_cases) == 190, (
    f"Unexpected high-cost loss population: {len(loss_cases)}"
)


# ------------------------------------------------------------
# 06.3 | FINANCIAL EXPOSURE
# ------------------------------------------------------------

exposure = pd.DataFrame({
    "Population": [
        "All sales lines",
        "High-cost sales lines",
        "High-cost loss lines"
    ],
    "Records": [
        len(df),
        len(high_cost),
        len(loss_cases)
    ],
    "Net_Revenue": [
        df["net_revenue"].sum(),
        high_cost["net_revenue"].sum(),
        loss_cases["net_revenue"].sum()
    ],
    "Total_Cost": [
        df["total_cost"].sum(),
        high_cost["total_cost"].sum(),
        loss_cases["total_cost"].sum()
    ],
    "Gross_Profit": [
        df["gross_profit"].sum(),
        high_cost["gross_profit"].sum(),
        loss_cases["gross_profit"].sum()
    ]
})

exposure["Record_Share_%"] = (
    exposure["Records"] / len(df) * 100
)

exposure["Revenue_Share_%"] = (
    exposure["Net_Revenue"]
    / df["net_revenue"].sum() * 100
)

exposure["Gross_Margin_%"] = (
    exposure["Gross_Profit"]
    / exposure["Net_Revenue"] * 100
)

print("FINANCIAL EXPOSURE")
display(exposure.round(2))


# ------------------------------------------------------------
# 06.4 | COMMERCIAL CONCENTRATION FUNCTION
# ------------------------------------------------------------

def summarize_loss_concentration(
    population,
    dimension
):
    summary = (
        population
        .groupby(dimension, dropna=False)
        .agg(
            Loss_Records=("gross_profit", "size"),
            Net_Revenue=("net_revenue", "sum"),
            Total_Cost=("total_cost", "sum"),
            Gross_Profit=("gross_profit", "sum")
        )
    )

    segment_base = (
        high_cost
        .groupby(dimension, dropna=False)
        .size()
        .rename("High_Cost_Records")
    )

    summary = summary.join(segment_base)

    summary["Loss_Rate_Within_High_Cost_%"] = (
        summary["Loss_Records"]
        / summary["High_Cost_Records"] * 100
    )

    summary["Share_Of_190_Loss_Lines_%"] = (
        summary["Loss_Records"]
        / len(loss_cases) * 100
    )

    return (
        summary
        .sort_values(
            ["Loss_Records", "Gross_Profit"],
            ascending=[False, True]
        )
        .round(2)
    )


# ------------------------------------------------------------
# 06.5 | COMMERCIAL SEGMENT ANALYSIS
# ------------------------------------------------------------

for dimension in [
    "category",
    "channel_name",
    "order_status"
]:
    print(
        f"\nLOSS CONCENTRATION BY {dimension.upper()}"
    )

    display(
        summarize_loss_concentration(
            loss_cases,
            dimension
        )
    )


# ------------------------------------------------------------
# 06.6 | RECONCILIATION CONTROLS
# ------------------------------------------------------------

assert (
    exposure.loc[2, "Records"]
    == len(loss_cases)
)

assert abs(
    exposure.loc[2, "Gross_Profit"]
    - loss_cases["gross_profit"].sum()
) < 0.01

print("\nDQ013 exposure reconciliation: PASS")


FINANCIAL EXPOSURE


,Population,Records,Net_Revenue,Total_Cost,Gross_Profit,Record_Share_%,Revenue_Share_%,Gross_Margin_%
0,All sales lines,23530,14937536.82,10442161.65,4495375.17,100.00,100.00,30.09
1,High-cost sales lines,2610,6123688.67,4289867.04,1833821.63,11.09,41.00,29.95
2,High-cost loss lines,190,267770.43,291182.61,-23412.18,0.81,1.79,-8.74



LOSS CONCENTRATION BY CATEGORY


,Loss_Records,Net_Revenue,Total_Cost,Gross_Profit,High_Cost_Records,Loss_Rate_Within_High_Cost_%,Share_Of_190_Loss_Lines_%
category,,,,,,,
Technology,54,81735.87,88452.33,-6716.46,605,8.93,28.42
Accessories,47,67031.96,72956.44,-5924.48,433,10.85,24.74
Office,31,42419.83,47136.84,-4717.01,464,6.68,16.32
Wellness,30,39675.92,42472.53,-2796.61,520,5.77,15.79
Home,28,36906.85,40164.47,-3257.62,588,4.76,14.74



LOSS CONCENTRATION BY CHANNEL_NAME


,Loss_Records,Net_Revenue,Total_Cost,Gross_Profit,High_Cost_Records,Loss_Rate_Within_High_Cost_%,Share_Of_190_Loss_Lines_%
channel_name,,,,,,,
E-commerce,62,86756.39,94964.23,-8207.84,819,7.57,32.63
Retail Store,56,77592.38,83391.28,-5798.90,779,7.19,29.47
Marketplace,37,51773.27,56003.26,-4229.99,455,8.13,19.47
Corporate Sales,28,40588.83,44598.77,-4009.94,376,7.45,14.74
Telephone Sales,7,11059.56,12225.07,-1165.51,181,3.87,3.68



LOSS CONCENTRATION BY ORDER_STATUS


,Loss_Records,Net_Revenue,Total_Cost,Gross_Profit,High_Cost_Records,Loss_Rate_Within_High_Cost_%,Share_Of_190_Loss_Lines_%
order_status,,,,,,,
Completed,167,234929.66,255870.96,-20941.30,2377,7.03,87.89
Partially Returned,12,18362.65,19708.88,-1346.23,118,10.17,6.32
Pending,6,7615.92,8146.19,-530.27,37,16.22,3.16
Cancelled,5,6862.20,7456.58,-594.38,78,6.41,2.63



DQ013 exposure reconciliation: PASS



---

## 07 | Product-Level Loss Concentration

**Objective:** Identify products with concentrated negative gross profit among high-cost transactions.

**DQ Control:** DQ013  
**Source:** BR001 Analytical Extract  
**Master Register:** `Nova_BR001_Data_Quality_Workbook.xlsx`

### METHODOLOGY

1. Group high-cost transactions by product.
2. Identify negative-profit records within each product.
3. Calculate product-level loss rates.
4. Measure each product's contribution to total recorded gross loss.
5. Rank products by financial exposure.
6. Reconcile the aggregated population and financial totals.

### VALIDATION BASELINE

- High-cost negative-profit lines: 190
- Recorded gross loss: USD -23,412.18

### INTERPRETATION

Product-level concentration identifies potential priorities for commercial investigation.

It does not establish whether the underlying transactions are erroneous.

### GOVERNANCE

- No transactions are excluded.
- No product values are modified.
- Commercial and Finance validation is required before remediation.


In [28]:

# ============================================================
# NOVA CORPORATION | BR001
# 07 | PRODUCT-LEVEL LOSS CONCENTRATION
# ============================================================

# Reuses the validated high_cost and loss_cases populations.

# ------------------------------------------------------------
# 07.1 | PRODUCT KEY VALIDATION
# ------------------------------------------------------------

product_key = "product_id"

if product_key not in high_cost.columns:
    raise ValueError(
        "product_id is missing from high_cost."
    )


# ------------------------------------------------------------
# 07.2 | HIGH-COST PRODUCT BASELINE
# ------------------------------------------------------------

product_base = (
    high_cost
    .groupby(product_key, dropna=False)
    .agg(
        High_Cost_Records=("gross_profit", "size"),
        High_Cost_Net_Revenue=("net_revenue", "sum")
    )
)


# ------------------------------------------------------------
# 07.3 | NEGATIVE-PROFIT PRODUCT POPULATION
# ------------------------------------------------------------

product_losses = (
    loss_cases
    .groupby(product_key, dropna=False)
    .agg(
        Loss_Records=("gross_profit", "size"),
        Loss_Net_Revenue=("net_revenue", "sum"),
        Loss_Gross_Profit=("gross_profit", "sum")
    )
)


# ------------------------------------------------------------
# 07.4 | PRODUCT-LEVEL METRICS
# ------------------------------------------------------------

product_review = (
    product_base
    .join(product_losses, how="left")
)

product_review[
    [
        "Loss_Records",
        "Loss_Net_Revenue",
        "Loss_Gross_Profit"
    ]
] = product_review[
    [
        "Loss_Records",
        "Loss_Net_Revenue",
        "Loss_Gross_Profit"
    ]
].fillna(0)

product_review["Loss_Rate_%"] = (
    product_review["Loss_Records"]
    / product_review["High_Cost_Records"] * 100
)

product_review["Share_Of_Total_Loss_%"] = (
    product_review["Loss_Gross_Profit"]
    / loss_cases["gross_profit"].sum() * 100
)


# ------------------------------------------------------------
# 07.5 | PRODUCT RANKING
# ------------------------------------------------------------

product_review = (
    product_review
    .query("Loss_Records > 0")
    .sort_values(
        ["Loss_Gross_Profit", "Loss_Records"],
        ascending=[True, False]
    )
)

print("TOP 15 PRODUCTS BY GROSS LOSS")

display(
    product_review.head(15).round(2)
)


# ------------------------------------------------------------
# 07.6 | PRODUCT-LEVEL RECONCILIATION
# ------------------------------------------------------------

print("\nPRODUCT-LEVEL RECONCILIATION")

print(
    "Products with losses:",
    len(product_review)
)

print(
    "Loss records:",
    int(product_review["Loss_Records"].sum())
)

print(
    "Total gross loss:",
    round(
        product_review["Loss_Gross_Profit"].sum(),
        2
    )
)

assert int(
    product_review["Loss_Records"].sum()
) == 190

assert abs(
    product_review["Loss_Gross_Profit"].sum()
    - (-23412.18)
) < 0.01

print("DQ013 product reconciliation: PASS")


TOP 15 PRODUCTS BY GROSS LOSS


,High_Cost_Records,High_Cost_Net_Revenue,Loss_Records,Loss_Net_Revenue,Loss_Gross_Profit,Loss_Rate_%,Share_Of_Total_Loss_%
product_id,,,,,,,
P0151,35,66357.50,20.0,28405.52,-2958.51,57.14,12.64
P0028,37,70771.21,11.0,19530.93,-1895.45,29.73,8.10
P0083,21,35791.53,10.0,14427.62,-1865.47,47.62,7.97
P0170,17,28495.57,7.0,10142.33,-1342.79,41.18,5.74
P0096,23,44371.22,7.0,10102.25,-1332.86,30.43,5.69
P0031,27,51961.93,5.0,9110.72,-1055.54,18.52,4.51
P0005,28,59972.86,7.0,9946.79,-1038.66,25.00,4.44
P0055,26,44609.88,7.0,9679.75,-967.64,26.92,4.13
P0029,25,36951.45,6.0,6907.39,-896.78,24.00,3.83



PRODUCT-LEVEL RECONCILIATION
Products with losses: 56
Loss records: 190
Total gross loss: -23412.18
DQ013 product reconciliation: PASS



---

## 08 | DQ013 — Investigation Conclusion

**Control:** DQ013 — Statistical Outlier Assessment  
**Control Status:** REVIEW — Pending Business Clarification
**Related Clarification:** CL-007 — OPEN 
**Master Register:** `Nova_BR001_Data_Quality_Workbook.xlsx`

### 08.1 | INVESTIGATION FINDINGS

The IQR assessment identified 2,610 positive high-total-cost outlier records.

These observations represent:

- 11.09% of the 23,530 sales lines.
- 41.00% of recorded net revenue.

Within this population:

- 190 sales lines have negative gross profit.
- Recorded gross loss totals USD -23,412.18.
- These records represent 0.81% of all sales lines.

### 08.2 | COMMERCIAL CONCENTRATION

The 190 negative-profit sales lines span:

- 56 products.
- All five sales channels.
- Multiple product categories.

Technology and Accessories account for 101 of the 190 negative-profit lines.

Product P0151 has the largest recorded gross loss:

- Gross loss: USD -2,958.51.
- Negative-profit lines: 20.
- Total high-cost lines: 35.

Of the 190 negative-profit lines, 167 have an order status of Completed.

### 08.3 | TECHNICAL VALIDATION

The investigation established the following:

- Statistical outlier populations were identified.
- Financial exposure was quantified.
- Commercial concentration was assessed.
- Product-level financial reconciliations passed.
- No source records were modified.

### 08.4 | INTERPRETATION & LIMITATIONS

Statistical outlier status does not establish a data-quality defect.

Negative gross profit may reflect legitimate pricing, discounting, or cost conditions.

These explanations have not been independently validated by the business.

Therefore, the investigation cannot authorize automatic correction or exclusion of the affected records.

### 08.5 | RECOMMENDED DISPOSITION

1. Retain statistical outliers pending business validation.
2. Do not remove, cap, winsorize, or automatically correct observations.
3. Request Finance and Commercial review of high-cost negative-margin transactions.
4. Prioritize products with concentrated financial exposure.
5. Keep confirmed numeric-rule violations under DQ007 / CL-005 as a separate issue.
6. Reassess KPI eligibility after receiving authorized clarification.

### 08.6 | CLARIFICATION HANDOFF

**Required Stakeholders:** Finance / Commercial

**Pending Decisions:**

- Confirm whether unusual transaction quantities are commercially legitimate.
- Validate the business circumstances of high-cost negative-margin transactions.
- Determine whether additional investigation or remediation is required.
- Confirm the appropriate treatment for downstream KPI calculations.

### CONTROL DISPOSITION

**Technical Investigation:** Completed  
**Financial Reconciliation:** Passed  
**Business Validation:** Pending  
**DQ013 Assessment:** REVIEW — Pending Business Clarification

The official status and subsequent decisions must be maintained in the master Data Quality workbook.

---
